## 1. Data Loading and Initial Inspection
Importing the school data and displaying the first five rows

In [1]:
import pandas as pd
from pathlib import Path

# Look for the data file in the places the notebook is usually run from:
# the Codes folder (Jupyter / VS Code), the repo root, or next to the notebook (e.g. uploaded to Colab)
file_name = 'مدارس_المملكة_تقييم_سنوي_شامل.csv'
candidates = [Path('../Data') / file_name, Path('Data') / file_name, Path(file_name)]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(f"Couldn't find {file_name} in ../Data, Data, or the current folder")

df = pd.read_csv(data_path)
display(df.head())

,المنطقة,الإدارة,اسم المدرسة,قدرات_1443,تحصيلي_1443,التقويم_الذاتي_1443,التقويم_الخارجي_1443,الفرق_بين_التقويمين_1443,قدرات_1444,تحصيلي_1444,...,التقويم_الخارجي_1446,الفرق_بين_التقويمين_1446,قدرات_1447,تحصيلي_1447,التقويم_الذاتي_1447,التقويم_الخارجي_1447,الفرق_بين_التقويمين_1447,نسبة تقدم القدرات (%),نسبة تقدم التحصيلي (%),تقييم هيئة التقويم والتعليم
0,الباحة,إدارة التعليم بمحافظة المخواة,الثانوية الاولى بالمخواة نظام المقررات,72.88,67.06,78.96,64.83,14.13,73.91,72.76,...,68.72,8.23,70.96,71.48,82.44,67.27,15.17,-2.63%,6.59%,متقدم
1,الباحة,الإدارة العامة للتعليم بمنطقة الباحة,ثانوية أروى بنت عبد المطلب بن هشام بربوع الصف...,65.26,64.90,71.53,56.62,14.91,70.30,61.07,...,65.29,6.06,66.25,67.15,72.89,60.47,12.42,1.52%,3.47%,منطلق
2,الباحة,الإدارة العامة للتعليم بمنطقة الباحة,ثانوية أم عبد الله بن عمر بن الخطاب بالجبل وا...,74.70,73.64,83.76,69.05,14.71,72.74,76.09,...,67.17,2.06,72.61,75.17,83.95,71.34,12.61,-2.8%,2.08%,متقدم
3,الباحة,الإدارة العامة للتعليم بمنطقة الباحة,ثانوية أم عمارة بالمندق - مقررات,68.72,71.90,83.40,63.13,20.27,73.38,70.19,...,69.98,8.28,78.83,66.12,73.88,67.20,6.68,14.71%,-8.04%,متقدم
4,الباحة,الإدارة العامة للتعليم بمنطقة الباحة,ثانوية أم هشام بنت حارثة بالأطاولة - مقررات,71.69,71.28,77.55,73.60,3.95,67.03,73.46,...,69.12,12.57,68.66,76.76,76.96,69.58,7.38,-4.23%,7.69%,متقدم


## 2. Data Formatting & Handling Missing Values
Removing incomplete records and converting percentage strings into workable decimals.

In [2]:
# Drop missing values and format percentage columns into decimals
df_cleaned = df.dropna().copy()

columns_to_clean = ['نسبة تقدم القدرات (%)', 'نسبة تقدم التحصيلي (%)']
for col in columns_to_clean:
    if df_cleaned[col].dtype == 'object':
        df_cleaned[col] = df_cleaned[col].str.replace('%', '').astype(float)

display(df_cleaned[columns_to_clean].head())

,نسبة تقدم القدرات (%),نسبة تقدم التحصيلي (%)
0,-2.63%,6.59%
1,1.52%,3.47%
2,-2.8%,2.08%
3,14.71%,-8.04%
4,-4.23%,7.69%


## 3. Feature Scaling (Normalization)
Compressing numerical features to a 0-1 scale using MinMaxScaler

In [3]:
# Scale numerical features to a standard range between 0 and 1
from sklearn.preprocessing import MinMaxScaler

features = ['قدرات_1447', 'تحصيلي_1447', 'التقويم_الخارجي_1447', 'التقويم_الذاتي_1447']
scaler = MinMaxScaler()
df_cleaned[features] = scaler.fit_transform(df_cleaned[features])

display(df_cleaned[features].head())

,قدرات_1447,تحصيلي_1447,التقويم_الخارجي_1447,التقويم_الذاتي_1447
0,0.542109,0.572114,0.521900,0.657232
1,0.452769,0.473143,0.382069,0.470818
2,0.573407,0.656457,0.605593,0.686707
3,0.691388,0.449600,0.520461,0.490142
4,0.498483,0.692800,0.569402,0.550264


## 4. Linear Regression
Splitting data and training a linear model to extract feature weights

In [4]:
# Split data and train Linear Regression to extract feature weights
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = df_cleaned[['قدرات_1447', 'التقويم_الخارجي_1447', 'التقويم_الذاتي_1447']]
y = df_cleaned['تحصيلي_1447']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

print("--- (Multiple Linear Regression) نتائج نموذج  ---")
print(f"نقطة التقاطع (w0): {model_lr.intercept_:.4f}")
print(f"وزن القدرات (w1): {model_lr.coef_[0]:.4f}")
print(f"وزن التقويم الخارجي (w2): {model_lr.coef_[1]:.4f}")
print(f"وزن التقويم الذاتي (w3): {model_lr.coef_[2]:.4f}")

--- (Multiple Linear Regression) نتائج نموذج  ---
نقطة التقاطع (w0): 0.0505
وزن القدرات (w1): -0.4194
وزن التقويم الخارجي (w2): 0.7999
وزن التقويم الذاتي (w3): 0.4827


## 5. Random Forest
Training a Random Forest model to calculate feature importance scores

In [5]:
# Train Random Forest to evaluate the importance of each feature
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(random_state=42)
model_rf.fit(X_train, y_train)

print("--- (Random Forest) أهمية العوامل في  ---")
for feature, importance in zip(X.columns, model_rf.feature_importances_):
    print(f"{feature}: {importance:.4f}")

--- (Random Forest) أهمية العوامل في  ---
قدرات_1447: 0.1405
التقويم_الخارجي_1447: 0.6787
التقويم_الذاتي_1447: 0.1807


## 6. XGBoost
Training an XGBoost model: the trees are built one after another, and each new tree learns to correct the errors left by the trees before it.

In [6]:
# Train XGBoost: trees are built in sequence, each one correcting the errors of the previous ones
from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    n_estimators=300,    # number of trees, built one after another
    learning_rate=0.05,  # how much of each tree's correction is kept
    max_depth=3,         # keep every tree small (a "weak learner")
    subsample=0.8,       # each tree trains on a random 80% of the schools
    random_state=42
)
model_xgb.fit(X_train, y_train)

print("--- (XGBoost) أهمية العوامل في  ---")
for feature, importance in zip(X.columns, model_xgb.feature_importances_):
    print(f"{feature}: {importance:.4f}")

--- (XGBoost) أهمية العوامل في  ---
قدرات_1447: 0.0719
التقويم_الخارجي_1447: 0.7516
التقويم_الذاتي_1447: 0.1765


## 7. Model Comparison
Evaluating the three models on the same test set (the 20% of schools never seen in training), then with 5-fold cross-validation for a fairer average. MAE and RMSE are converted back to Tahsili score points.

In [7]:
# Compare the three models on the same test set and with 5-fold cross-validation
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

# Tahsili was scaled to 0-1, so multiplying an error by its original range turns it back into score points
tahsili_range = scaler.data_range_[features.index('تحصيلي_1447')]

models = {
    'Linear Regression': model_lr,
    'Random Forest': model_rf,
    'XGBoost': model_xgb,
}
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    cv_r2 = cross_val_score(model, X, y, cv=kfold, scoring='r2')
    results.append({
        'Model': name,
        'MAE (points)': mean_absolute_error(y_test, y_pred) * tahsili_range,
        'RMSE (points)': np.sqrt(mean_squared_error(y_test, y_pred)) * tahsili_range,
        'R2 (test set)': r2_score(y_test, y_pred),
        'R2 (5-fold CV)': cv_r2.mean(),
    })

display(pd.DataFrame(results).round(3))

,Model,MAE (points),RMSE (points),R2 (test set),R2 (5-fold CV)
0,Linear Regression,2.913,3.633,0.666,0.677
1,Random Forest,3.069,3.832,0.628,0.633
2,XGBoost,2.960,3.694,0.654,0.669
